<a href="https://colab.research.google.com/github/dee431/Strategic-Planning-Requirement-Analysis-for-a-Virtual-NLP-Chatbot/blob/main/Strategic_Planning_%26_Requirement_Analysis_for_a_Virtual_NLP_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cell 1: Imports and Setup**

In [1]:
# ============================================
# GLOWGUIDE AI - Virtual Beauty & Wellness Chatbot (Prototype)
# Week 1 Strategic Plan Implementation
# ============================================

# This notebook implements a rule-based NLP chatbot with:
# - Intent recognition via keyword matching
# - Entity extraction for skin types, concerns, product types, ingredients
# - Simple dialogue management (state tracking)
# - Response generation based on user profile and intent

# All required libraries are pre-installed in Colab
import re
import random
from typing import Dict, List, Tuple, Optional

# **Cell 2: Knowledge Base and Data Structures**

In [2]:
# ============================================
# Define Knowledge Base and Data Structures
# ============================================

# Skin types
SKIN_TYPES = ["oily", "dry", "combination", "sensitive", "normal"]

# Common concerns
CONCERNS = ["acne", "aging", "dullness", "redness", "dark spots", "pores", "texture"]

# Product types
PRODUCT_TYPES = ["cleanser", "toner", "serum", "moisturizer", "sunscreen", "exfoliator", "mask"]

# Ingredients knowledge base (simplified)
INGREDIENT_INFO = {
    "niacinamide": "Niacinamide (Vitamin B3) helps reduce inflammation, minimize pores, and regulate oil production. Suitable for most skin types, especially oily and acne-prone.",
    "retinol": "Retinol (Vitamin A) promotes cell turnover, reduces fine lines and wrinkles. Best for aging concerns; start with low concentration and use at night.",
    "hyaluronic acid": "Hyaluronic acid is a powerful humectant that attracts moisture, keeping skin hydrated and plump. Good for all skin types, especially dry.",
    "salicylic acid": "Salicylic acid (BHA) exfoliates inside pores, making it excellent for acne and blackheads. Ideal for oily and combination skin.",
    "vitamin c": "Vitamin C is an antioxidant that brightens skin, fades dark spots, and boosts collagen. Use in the morning under sunscreen.",
    "glycolic acid": "Glycolic acid (AHA) exfoliates the skin surface, improving texture and radiance. Avoid if you have very sensitive skin.",
    "ceramides": "Ceramides strengthen the skin barrier and lock in moisture. Great for dry, sensitive, or compromised skin.",
    "zinc oxide": "Zinc oxide is a physical sunscreen ingredient that provides broad-spectrum UV protection. Non-irritating and suitable for sensitive skin."
}

# Sample product recommendations by skin type and concern (simplified)
ROUTINE_TEMPLATES = {
    "oily": {
        "cleanser": "Use a gentle foaming cleanser with salicylic acid or tea tree oil.",
        "toner": "Apply an alcohol-free toner with niacinamide to control oil.",
        "serum": "Use a lightweight serum with niacinamide or hyaluronic acid.",
        "moisturizer": "Choose an oil-free, gel-based moisturizer.",
        "sunscreen": "Use a mattifying, broad-spectrum SPF 30+ sunscreen."
    },
    "dry": {
        "cleanser": "Use a creamy, hydrating cleanser without sulfates.",
        "toner": "Apply a hydrating toner with hyaluronic acid or glycerin.",
        "serum": "Use a hydrating serum with hyaluronic acid and ceramides.",
        "moisturizer": "Choose a rich, cream-based moisturizer with ceramides and squalane.",
        "sunscreen": "Use a moisturizing sunscreen with SPF 30+."
    },
    "combination": {
        "cleanser": "Use a gentle, pH-balanced cleanser.",
        "toner": "Apply a balancing toner with niacinamide.",
        "serum": "Use a lightweight hydrating serum; target oily areas with niacinamide.",
        "moisturizer": "Use a lotion that is hydrating but not heavy.",
        "sunscreen": "Use a broad-spectrum SPF 30+ that is not greasy."
    },
    "sensitive": {
        "cleanser": "Use a fragrance-free, creamy cleanser with minimal ingredients.",
        "toner": "Use a soothing, alcohol-free toner with chamomile or aloe.",
        "serum": "Use a calming serum with centella asiatica or ceramides.",
        "moisturizer": "Choose a hypoallergenic, fragrance-free moisturizer.",
        "sunscreen": "Use a mineral (zinc oxide/titanium dioxide) sunscreen."
    },
    "normal": {
        "cleanser": "Use a gentle, non-stripping cleanser.",
        "toner": "Apply a hydrating toner with antioxidants.",
        "serum": "Use a vitamin C serum for antioxidant protection.",
        "moisturizer": "Use a light lotion or gel-cream.",
        "sunscreen": "Use any broad-spectrum SPF 30+."
    }
}

# Wellness tips
WELLNESS_TIPS = [
    "Drink at least 8 glasses of water daily to keep skin hydrated.",
    "Aim for 7-9 hours of quality sleep for skin repair and regeneration.",
    "Manage stress through meditation or yoga; high cortisol can cause breakouts.",
    "Eat a balanced diet rich in antioxidants (berries, leafy greens) and omega-3s.",
    "Exercise regularly to improve blood circulation and skin glow."
]

# **Cell 3: Intent Recognition Functions**

In [3]:
# ============================================
# Intent Recognition Functions
# ============================================

def recognize_intent(user_input: str) -> str:
    """
    Classify user input into one of the predefined intents.
    Uses keyword/regex matching for simplicity.
    """
    text = user_input.lower().strip()

    # Greeting
    if re.search(r'\b(hi|hello|hey|good\s*(morning|afternoon|evening)|howdy)\b', text):
        return "greeting"

    # Asking for skin type
    if re.search(r'\b(what|which)\s+(is\s+)?my\s+skin\s+type\b', text):
        return "ask_skin_type"

    # Providing skin type
    for st in SKIN_TYPES:
        if re.search(rf'\b{st}\b', text):
            # Check if it's an explicit statement like "I have oily skin"
            if re.search(rf'\b(i\s+have|my\s+skin\s+is|i\s+am)\s+{st}', text) or st in text:
                return "provide_skin_type"

    # Asking for routine
    if re.search(r'\b(routine|regimen|steps|what\s+should\s+i\s+use|products)\b', text):
        return "ask_routine"

    # Asking for product recommendation
    if re.search(r'\b(recommend|suggest|best\s+product|which\s+product)\b', text):
        return "ask_product_recommendation"

    # Asking about ingredient
    for ing in INGREDIENT_INFO.keys():
        if re.search(rf'\b{ing}\b', text):
            if re.search(r'\b(what|tell\s+me|about|is|benefits?)\b', text):
                return "ask_ingredient"

    # Wellness advice
    if re.search(r'\b(wellness|health|diet|sleep|stress|hydration|water|exercise)\b', text):
        return "ask_wellness"

    # Mental wellness
    if re.search(r'\b(stress|anxious|sad|depressed|overwhelmed|mental\s+health)\b', text):
        return "ask_mental_health"

    # Thanks
    if re.search(r'\b(thank|thanks|thx|appreciate)\b', text):
        return "thanks"

    # Goodbye
    if re.search(r'\b(bye|goodbye|see\s+you|exit|quit)\b', text):
        return "goodbye"

    # Fallback
    return "fallback"

# **Cell 4: Entity Extraction Functions**

In [4]:
# ============================================
# Entity Extraction Functions
# ============================================

def extract_skin_type(text: str) -> Optional[str]:
    """Extract skin type from user input."""
    text_lower = text.lower()
    for st in SKIN_TYPES:
        if re.search(rf'\b{st}\b', text_lower):
            return st
    return None

def extract_concern(text: str) -> Optional[str]:
    """Extract common skin concerns."""
    text_lower = text.lower()
    for concern in CONCERNS:
        if re.search(rf'\b{concern}\b', text_lower):
            return concern
    return None

def extract_ingredient(text: str) -> Optional[str]:
    """Extract ingredient name from user input."""
    text_lower = text.lower()
    for ing in INGREDIENT_INFO.keys():
        if re.search(rf'\b{ing}\b', text_lower):
            return ing
    return None

def extract_product_type(text: str) -> Optional[str]:
    """Extract product type from user input."""
    text_lower = text.lower()
    for pt in PRODUCT_TYPES:
        if re.search(rf'\b{pt}\b', text_lower):
            return pt
    return None

# **Cell 5: Dialogue State Management**

In [5]:
# ============================================
# Dialogue State Management
# ============================================

class ChatbotState:
    """Maintains conversation context and user profile."""
    def __init__(self):
        self.user_profile = {
            "skin_type": None,
            "concerns": [],
            "name": None
        }
        self.current_question = None  # what the bot asked last
        self.last_intent = None
        self.history = []  # list of (user, bot) tuples

    def add_to_history(self, user_msg, bot_msg):
        self.history.append((user_msg, bot_msg))

    def get_profile_summary(self) -> str:
        if self.user_profile["skin_type"]:
            return f"Your skin type is {self.user_profile['skin_type']}."
        return "I don't have your skin type yet."

# **Cell 6: Response Generation Functions**

In [6]:
# ============================================
# Response Generation
# ============================================

def generate_greeting(state: ChatbotState) -> str:
    """Generate greeting and ask for skin type if unknown."""
    if not state.user_profile["skin_type"]:
        return random.choice([
            "Hello! I'm GlowGuide AI, your virtual beauty and wellness advisor. To personalise my advice, could you tell me your skin type? (oily, dry, combination, sensitive, normal)",
            "Hi there! I'm here to help with your beauty and wellness questions. First, what's your skin type?",
            "Welcome! I can help with skincare routines, product recommendations, and wellness tips. Let's start: what's your skin type?"
        ])
    else:
        return f"Hello again! I see you have {state.user_profile['skin_type']} skin. How can I assist you today?"

def generate_skin_type_response(state: ChatbotState, skin_type: str) -> str:
    """Store skin type and provide a brief comment."""
    state.user_profile["skin_type"] = skin_type
    comments = {
        "oily": "Oily skin tends to produce excess sebum, especially in the T-zone. We'll focus on oil control without over-drying.",
        "dry": "Dry skin often feels tight and may have flakiness. We'll prioritise deep hydration and barrier repair.",
        "combination": "Combination skin has an oily T-zone and dry cheeks. We'll balance both areas.",
        "sensitive": "Sensitive skin reacts easily to products. We'll use gentle, fragrance-free options.",
        "normal": "Normal skin is well-balanced. We'll maintain health with antioxidants and SPF."
    }
    return f"Got it! {comments.get(skin_type, 'Thanks for sharing.')} Would you like a personalised skincare routine?"

def generate_routine(state: ChatbotState) -> str:
    """Generate a step-by-step routine based on skin type."""
    skin_type = state.user_profile["skin_type"]
    if not skin_type:
        return "I need to know your skin type first. Could you tell me? (oily, dry, combination, sensitive, normal)"

    template = ROUTINE_TEMPLATES.get(skin_type, ROUTINE_TEMPLATES["normal"])
    routine = "Here's a suggested AM/PM routine for your skin type:\n"
    for step, advice in template.items():
        routine += f"\n• {step.capitalize()}: {advice}"
    routine += "\n\nDo you have any specific concerns (e.g., acne, aging, dullness)?"
    return routine

def generate_product_recommendation(state: ChatbotState, user_input: str) -> str:
    """Provide product recommendation based on extracted product type and skin type."""
    product_type = extract_product_type(user_input)
    skin_type = state.user_profile["skin_type"]
    if not skin_type:
        return "First, please tell me your skin type so I can recommend suitable products."
    if not product_type:
        return "Which product type are you looking for? (cleanser, serum, moisturizer, sunscreen, etc.)"

    # Simple mapping: provide general advice from ROUTINE_TEMPLATES
    if product_type in ROUTINE_TEMPLATES[skin_type]:
        advice = ROUTINE_TEMPLATES[skin_type][product_type]
        return f"For {skin_type} skin, I recommend: {advice}"
    else:
        return f"I don't have specific {product_type} advice yet, but I can suggest looking for products suited to {skin_type} skin."

def generate_ingredient_info(ingredient: str) -> str:
    """Return information about a specific ingredient."""
    return INGREDIENT_INFO.get(ingredient, "I don't have information on that ingredient yet. Please consult a dermatologist or reliable source.")

def generate_wellness_tip() -> str:
    """Provide a random wellness tip."""
    return random.choice(WELLNESS_TIPS)

def generate_mental_wellness_response(state: ChatbotState, user_input: str) -> str:
    """Provide supportive response for mental wellness queries."""
    return ("I'm here to support you. Remember that stress and emotions can affect your skin. "
            "Try to practice self-care, get enough sleep, and talk to someone you trust. "
            "If you're feeling overwhelmed, please consider reaching out to a mental health professional. "
            "You're not alone. 💙")

def generate_fallback_response() -> str:
    """Provide a fallback when intent is not recognized."""
    return random.choice([
        "I'm not sure I understood that. Could you rephrase? I can help with skincare routines, product recommendations, ingredients, and wellness tips.",
        "I'm still learning! Could you ask me about skin types, routines, or specific ingredients?",
        "Sorry, I didn't catch that. You can ask me for a routine, product advice, or general wellness tips."
    ])

def generate_response(state: ChatbotState, user_input: str) -> str:
    """Main response generation function based on intent and entities."""
    intent = recognize_intent(user_input)
    state.last_intent = intent

    # Extract entities
    skin_type = extract_skin_type(user_input)
    ingredient = extract_ingredient(user_input)

    # Handle each intent
    if intent == "greeting":
        return generate_greeting(state)

    elif intent == "provide_skin_type" and skin_type:
        return generate_skin_type_response(state, skin_type)

    elif intent == "ask_skin_type":
        return "Your skin type is usually determined by how much oil your skin produces. Common types: oily, dry, combination, sensitive, normal. Do you know yours? If not, I can help you figure it out."

    elif intent == "ask_routine":
        return generate_routine(state)

    elif intent == "ask_product_recommendation":
        return generate_product_recommendation(state, user_input)

    elif intent == "ask_ingredient" and ingredient:
        return generate_ingredient_info(ingredient)

    elif intent == "ask_wellness":
        return generate_wellness_tip()

    elif intent == "ask_mental_health":
        return generate_mental_wellness_response(state, user_input)

    elif intent == "thanks":
        return "You're welcome! I'm happy to help. Anything else you'd like to know?"

    elif intent == "goodbye":
        return "Goodbye! Take care of your skin and wellness. Feel free to come back anytime. 👋"

    else:
        return generate_fallback_response()

# **Cell 7: Main Chat Loop (Interactiv**

In [7]:
# ============================================
# Main Chat Loop (Interactive)
# ============================================

def run_chatbot():
    """Run the chatbot in an interactive loop (works in Colab)."""
    print("="*60)
    print("GlowGuide AI - Virtual Beauty & Wellness Chatbot")
    print("Type 'quit' or 'exit' to end the conversation.")
    print("="*60)

    state = ChatbotState()

    # Initial greeting
    bot_msg = generate_greeting(state)
    print(f"Bot: {bot_msg}")
    state.add_to_history("", bot_msg)

    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue

        if user_input.lower() in ['quit', 'exit', 'bye', 'goodbye']:
            bot_msg = "Goodbye! Take care of your skin and wellness. Feel free to come back anytime. 👋"
            print(f"Bot: {bot_msg}")
            state.add_to_history(user_input, bot_msg)
            break

        bot_msg = generate_response(state, user_input)
        print(f"Bot: {bot_msg}")
        state.add_to_history(user_input, bot_msg)

# **Cell 8: Demo and Testing (Automated)**

In [8]:
# ============================================
# Demo and Testing (Automated)
# ============================================

if __name__ == "__main__":
    # For interactive use, uncomment the next line:
    # run_chatbot()

    # For automated demonstration, we'll simulate a conversation:
    print("Automated Demo (simulated conversation):\n")
    state = ChatbotState()

    # Simulate a conversation
    test_messages = [
        "Hello",
        "I have oily skin",
        "Can you give me a routine?",
        "What about niacinamide?",
        "Recommend a moisturizer",
        "Any wellness tips?",
        "Thank you",
        "Bye"
    ]

    # Initial greeting
    bot_msg = generate_greeting(state)
    print(f"Bot: {bot_msg}")

    for msg in test_messages:
        print(f"User: {msg}")
        bot_msg = generate_response(state, msg)
        print(f"Bot: {bot_msg}\n")

    print("\n--- Conversation History ---")
    for i, (user, bot) in enumerate(state.history):
        if user:
            print(f"{i+1}. User: {user}")
        print(f"   Bot: {bot}\n")

Automated Demo (simulated conversation):

Bot: Hello! I'm GlowGuide AI, your virtual beauty and wellness advisor. To personalise my advice, could you tell me your skin type? (oily, dry, combination, sensitive, normal)
User: Hello
Bot: Welcome! I can help with skincare routines, product recommendations, and wellness tips. Let's start: what's your skin type?

User: I have oily skin
Bot: Got it! Oily skin tends to produce excess sebum, especially in the T-zone. We'll focus on oil control without over-drying. Would you like a personalised skincare routine?

User: Can you give me a routine?
Bot: Here's a suggested AM/PM routine for your skin type:

• Cleanser: Use a gentle foaming cleanser with salicylic acid or tea tree oil.
• Toner: Apply an alcohol-free toner with niacinamide to control oil.
• Serum: Use a lightweight serum with niacinamide or hyaluronic acid.
• Moisturizer: Choose an oil-free, gel-based moisturizer.
• Sunscreen: Use a mattifying, broad-spectrum SPF 30+ sunscreen.

Do yo